# 🎭 Face Swap 免费 Colab 教程

**适用平台：** Google Colab (免费 T4 GPU)

**本教程功能：** 将源人物（source）的脸部替换到目标视频（target）中，生成换脸视频。

## ⚠️ 重要前提

1. 需要上传 `cloud_gpu_faceswap_upload.tar.gz`（由 `scripts/00_make_upload_package.sh` 在本地 Mac 生成）
2. Colab 会话有时间限制，建议准备 2–3 小时完成全流程
3. 训练时间越长，换脸效果越好（3000次迭代约 30–60 分钟）

## Step 0 — 检查 GPU 环境

⏱ 预计时间：几秒钟

确认 Colab 已分配到 NVIDIA GPU（免费版通常是 T4）。

In [ ]:
!nvidia-smi

In [ ]:
import os
ROOT = '/content/faceswap_work'
os.makedirs(ROOT, exist_ok=True)
print('工作目录:', ROOT)

## Step 1 — 上传打包文件

⏱ 预计时间：1–5 分钟（取决于上传速度）

从 Mac 上传由 `scripts/00_make_upload_package.sh` 生成的 `cloud_gpu_faceswap_upload.tar.gz`。

**如何生成上传包（Mac 本地执行）：**
```bash
cd cloud_gpu_faceswap
bash scripts/00_make_upload_package.sh
```
输出的文件路径会打印在终端，将该文件上传到这里。

In [ ]:
from google.colab import files

uploaded = files.upload()
bundle = next((name for name in uploaded if name.endswith('.tar.gz')), None)
if bundle is None:
    raise RuntimeError('请上传 cloud_gpu_faceswap_upload.tar.gz 文件')

# 解压到工作目录
!rm -rf /content/faceswap_work/cloud_gpu_faceswap /content/faceswap_work/source_faces
!tar -xzf "{bundle}" -C /content/faceswap_work

# 显示解压后的文件结构（确认完整性）
!find /content/faceswap_work -maxdepth 3 -type f | sort | sed -n '1,80p'

## Step 2 — 安装 Faceswap 工具

⏱ 预计时间：10–20 分钟（Colab 免费版较慢）

从 GitHub 克隆 deepfakes/faceswap 仓库，并安装 NVIDIA CUDA 依赖。

**包括以下步骤：**
- `apt-get update` 更新系统包
- 安装 `ffmpeg`、`git`、`python3-venv`
- 克隆 faceswap 仓库
- 创建 Python 虚拟环境
- 安装 `requirements_nvidia.txt`（CUDA、cuDNN、TensorFlow 等）

⚠️ 如果 GitHub 访问被限制（国内常见），可能需要配置代理或使用镜像。

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap

# 安装系统依赖（root 权限在 Colab 中默认可用）
apt-get update -y
apt-get install -y ffmpeg git python3-venv

# 克隆 faceswap（已有则跳过）
mkdir -p tools
if [ ! -d tools/faceswap/.git ]; then
  echo '正在克隆 faceswap 仓库 ...'
  git clone https://github.com/deepfakes/faceswap.git tools/faceswap
else
  echo 'faceswap 仓库已存在，跳过克隆'
fi

# 创建 Python 虚拟环境
cd tools/faceswap
python3 -m venv .venv
source .venv/bin/activate

# 升级 pip 和安装依赖
python -m pip install --upgrade pip setuptools wheel
python -m pip install -r requirements/requirements_nvidia.txt

# 验证安装成功
python faceswap.py -h >/dev/null
echo '\n✅ faceswap 安装完成！'

## Step 3 — 准备素材并提取人脸

⏱ 预计时间：5–15 分钟（取决于视频帧数和 GPU 速度）

本步骤执行两个操作：
1. **提取源图片帧**：将 `source_faces/` 目录下的源人物照片裁剪为人脸
2. **提取目标视频帧**：将目标视频分解为帧，并裁剪所有人脸

⚠️ **建议在训练前人工检查提取结果**：如果发现错误的人脸图，删除对应文件。

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap

# 准备工作区（复制源图片 + 拆解目标视频）
bash scripts/02_prepare_workspace.sh

# 提取源和目标视频中的人脸
bash scripts/03_extract_faces.sh

## Step 4 — 人工检查提取的人脸

查看下方生成的缩略图拼图，确认：
- **Source faces**：全部是源人物的脸（用于学习源身份）
- **Target faces**：全部是目标视频中需要被替换的脸

如果发现错误人脸：
1. 在文件管理器中找到 `/content/faceswap_work/cloud_gpu_faceswap/workspace/`
2. 进入 `source_faces_extract/` 或 `target_faces_extract/` 目录
3. 删除错误的人脸图

✅ 确认无误后，进入下一步训练。

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

def contact_sheet(folder, title, thumb=128, cols=8):
    """生成缩略图拼图，方便快速检查人脸提取结果"""
    paths = sorted(Path(folder).glob('*'))[:64]
    if not paths:
        print('无图片:', folder)
        return
    rows = (len(paths) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * thumb, rows * (thumb + 22)), 'white')
    draw = ImageDraw.Draw(sheet)
    for idx, path in enumerate(paths):
        img = Image.open(path).convert('RGB')
        img = ImageOps.contain(img, (thumb, thumb))
        x = (idx % cols) * thumb
        y = (idx // cols) * (thumb + 22)
        sheet.paste(img, (x, y))
        draw.text((x + 2, y + thumb + 2), path.name[:18], fill=(0, 0, 0))
    print(title)
    display(sheet)

base = '/content/faceswap_work/cloud_gpu_faceswap/workspace'
contact_sheet(f'{base}/source_faces_extract', '【源人物】人脸提取结果（Source Faces）')
contact_sheet(f'{base}/target_faces_extract', '【目标人物】人脸提取结果（Target Faces）')

## Step 5 — 训练模型

⏱ 预计时间：
- **1000 次迭代**：15–30 分钟（快速测试）
- **3000 次迭代**：45–90 分钟（推荐质量）
- **5000 次迭代**：90–180 分钟（高质量）

**迭代次数越多，换脸越逼真，但需要更长运行时间。**

Colab 免费版可能因超时断开连接。建议：
- 设置中间断点保存（脚本每 250 次迭代保存预览图）
- 如需长时训练，可将 `workspace/model` 目录同步到 Google Drive

**调整迭代次数（默认 3000）：**
```bash
ITERATIONS=5000 bash scripts/04_train_preview.sh
```

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap

# 训练模型（可修改 ITERATIONS 环境变量调整迭代次数）
# 推荐：Colab 免费版用 3000；Colab Pro 可用 5000+
ITERATIONS=3000 bash scripts/04_train_preview.sh

## Step 6 — 转换并下载换脸视频

⏱ 预计时间：3–10 分钟

使用训练好的模型将源脸替换到目标视频帧，然后合成为 MP4。

输出文件：`output/faceswap_test.mp4`

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap

# 执行换脸转换
bash scripts/05_convert_test.sh

# 确认输出文件
ls -lh output/faceswap_test.mp4

In [ ]:
# 下载换脸视频
from google.colab import files
files.download('/content/faceswap_work/cloud_gpu_faceswap/output/faceswap_test.mp4')

---

## ❓ 常见问题（FAQ）

### Q1: Colab 断连了怎么办？
**A：** Colab 免费版约 90 分钟无活动会自动断开。解决建议：
- 在代码单元格中使用 `time.sleep()` 或播放音乐保持活跃
- 将 `workspace/model` 同步到 Google Drive（需要先挂载）
- 训练分多次进行，每次训练后保存 checkpoint

### Q2: GitHub 克隆失败（国内网络）
**A：** 配置代理或使用镜像：
```bash
git clone https://github.com.cnpmjs.org/deepfakes/faceswap.git tools/faceswap
```

### Q3: GPU 显存不足（OOM）
**A：** 降低 batch size（脚本默认 `bs=8`），改为 `bs=4` 或 `bs=2`：
编辑 `scripts/04_train_preview.sh`，将 `-bs 8` 改为 `-bs 4`。

### Q4: 人脸提取结果很差/没有提取到人脸
**A：**
- 检查源图片质量（建议高清、正面照）
- 目标视频中的人脸被遮挡（如戴口罩、侧脸）可能导致提取失败
- 确认使用的是 `s3fd` 检测器（脚本默认），它对各类场景较为鲁棒

### Q5: 如何提高换脸质量？
**A：**
- 增加训练迭代次数（3000 → 5000 → 10000）
- 提供更多源人物照片（建议 15–30 张）
- 使用更高质量的源图片（高清、正面、表情丰富）
- 尝试不同模型类型（`-t original` 可改为 `-t dfl_sae`）

### Q6: 如何换到完整视频？
**A：** 在验证测试视频效果满意后，运行：
```bash
bash scripts/06_convert_full_if_approved.sh
```
该脚本会使用完整目标视频（`input/target_normalized.mp4`）进行转换。